<a href="https://colab.research.google.com/github/fatimaibadovva-a11y/matrix_academy/blob/main/smart_waiter_bot_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-text-splitters

In [2]:
import os
import re

from langchain_text_splitters import MarkdownHeaderTextSplitter

import warnings

warnings.filterwarnings("ignore")

In [4]:
with open("narxana_restoran.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

print(markdown_text[:2000])

# Narxana Restoranlar Şəbəkəsi

> **Qeyd:** *Bu sənəd tədris məqsədilə hazırlanmış uydurma restoran haqqındadır. Bütün adlar, ünvanlar, telefon nömrələri, şəxslər və tərəfdaş şirkətlər uydurmadır.*

## Ümumi məlumat

**Narxana** 2009-cu ildə Bakıda açılmış Azərbaycan və Qafqaz mətbəxi restoranlar şəbəkəsidir. Restoranın adı *"nar"* və *"xana"* (ev, məkan) sözlərindən yaranıb və *"narın evi"* mənasını verir. Nar Azərbaycan mədəniyyətində bolluğun, bərəkətin və süfrə mədəniyyətinin simvoludur. Buna görə də narı həm loqomuzda, həm də menyumuzun bir çox yeməyində görə bilərsiniz.

Hazırda şəbəkənin 6 şəhərdə 9 filialı, 320-dən çox əməkdaşı var və hər gün orta hesabla 4 500 qonağa xidmət göstərir. Narxananın əsas prinsipi belədir: ***"Nənə reseptləri, müasir təqdimat."***

- **Mətbəx növü:** Azərbaycan, Qafqaz, qismən Aralıq dənizi mətbəxi
- **Qiymət səviyyəsi:** orta (bir nəfər üçün orta hesab 25–40 AZN)
- **Halal:** Bütün ət məhsulları halal sertifikatlıdır
- **Alkoqol:** Yalnız İçərişəhə

In [6]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

chunks = markdown_splitter.split_text(markdown_text)

print("Number of chunks:", len(chunks))

Number of chunks: 196


In [7]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
processed_chunks = []

def count_tokens(text):
    return len(
        embedding_model.tokenizer.encode(
            text,
            add_special_tokens=True
        )
    )

def split_long_line(line, prefix, max_tokens=256):
    words = line.split()
    parts = []
    current_words = []

    for word in words:
        candidate = " ".join(current_words + [word])

        if count_tokens(prefix + candidate) <= max_tokens:
            current_words.append(word)
        else:
            if current_words:
                parts.append(" ".join(current_words))
            current_words = [word]

    if current_words:
        parts.append(" ".join(current_words))

    return parts


for chunk in chunks:
    metadata = chunk.metadata

    path = " > ".join(
        str(value)
        for value in metadata.values()
        if value
    )

    prefix = f"Path: {path}\n\n"

    lines = [
        line for line in chunk.page_content.splitlines()
        if line.strip()
    ]

    current_lines = []

    for line in lines:

        if count_tokens(prefix + line) > 256:

            if current_lines:
                processed_chunks.append(
                    prefix + "\n".join(current_lines)
                )
                current_lines = []

            long_line_parts = split_long_line(
                line,
                prefix,
                max_tokens=256
            )

            for part in long_line_parts:
                processed_chunks.append(prefix + part)

        else:
            candidate = "\n".join(current_lines + [line])

            if count_tokens(prefix + candidate) <= 256:
                current_lines.append(line)

            else:
                if current_lines:
                    processed_chunks.append(
                        prefix + "\n".join(current_lines)
                    )

                current_lines = [line]

    if current_lines:
        processed_chunks.append(
            prefix + "\n".join(current_lines)
        )

print("Total processed chunks:", len(processed_chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (170 > 128). Running this sequence through the model will result in indexing errors


Total processed chunks: 202


In [11]:
token_lengths = [
    count_tokens(chunk)
    for chunk in processed_chunks
]

print("Number of chunks:", len(processed_chunks))
print("Maximum token length:", max(token_lengths))

Number of chunks: 202
Maximum token length: 246


In [12]:
embeddings = embedding_model.encode(
    processed_chunks,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Number of embeddings: 202
Embedding dimension: 384


In [13]:
!pip install -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.3 MB/s eta 0:00:00


In [14]:
import faiss
import numpy as np

In [15]:
embedding_matrix = np.array(embeddings).astype("float32")

print("Shape:", embedding_matrix.shape)

Shape: (202, 384)


In [16]:
embedding_matrix = np.array(embeddings).astype("float32")

# Cosine similarity üçün embedding-ləri normalize et
faiss.normalize_L2(embedding_matrix)

dimension = embedding_matrix.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embedding_matrix)

print("Number of vectors in index:", index.ntotal)


Number of vectors in index: 202


In [17]:
menu_chunks = []

for chunk in processed_chunks:
    path_line = chunk.split("\n", 1)[0]

    if " > Menyu" in path_line:
        menu_chunks.append(chunk)

print("Total menu chunks:", len(menu_chunks))


Total menu chunks: 159


In [18]:
menu_embeddings = embedding_model.encode(
    menu_chunks,
    show_progress_bar=True
)

print("Number of menu embeddings:", len(menu_embeddings))
print("Embedding dimension:", len(menu_embeddings[0]))

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Number of menu embeddings: 159
Embedding dimension: 384


In [19]:
menu_embedding_matrix = np.array(menu_embeddings).astype("float32")

# Embedding-ləri normalize et
faiss.normalize_L2(menu_embedding_matrix)

menu_dimension = menu_embedding_matrix.shape[1]

# Cosine similarity üçün Inner Product
menu_index = faiss.IndexFlatIP(menu_dimension)

menu_index.add(menu_embedding_matrix)

print("Number of vectors in menu index:", menu_index.ntotal)

Number of vectors in menu index: 159


In [20]:
def get_chunk_path(chunk):
    return chunk.split("\n", 1)[0].replace("Path: ", "").strip()


def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zəğıöşüç0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def add_same_path_chunks(results, matched_chunk):
    matched_path = get_chunk_path(matched_chunk)

    for chunk in menu_chunks:
        if get_chunk_path(chunk) == matched_path and chunk not in results:
            results.append(chunk)


def menu_retriever(query, k=5):
    query_embedding = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    search_k = min(max(k * 3, 10), len(menu_chunks))
    scores, indices = menu_index.search(query_embedding, search_k)

    results = []
    query_normalized = normalize_text(query)
    query_words = set(query_normalized.split())

    # 1. Konkret yemək adını tap və həmin yeməyin bütün chunk-larını əlavə et
    for chunk in menu_chunks:
        for line in chunk.split("\n"):
            if line.startswith("#### "):
                food_name = line.replace("#### ", "", 1).strip()
                food_words = [
                    word for word in normalize_text(food_name).split()
                    if len(word) >= 3
                ]

                matching_words = [
                    word for word in food_words
                    if any(word in qword or qword in word for qword in query_words)
                ]

                if len(matching_words) >= 2:
                    add_same_path_chunks(results, chunk)

    # 2. Kateqoriya və etiket uyğunluğu
    keywords = [
        "əsas yeməklər", "plovlar", "şorbalar", "salatlar", "desertlər",
        "səhər yeməkləri", "imza yeməkləri", "qlütensiz", "vegan",
        "vegetarian", "imza yeməyi"
    ]

    for keyword in keywords:
        keyword_words = normalize_text(keyword).split()
        if any(word in query_normalized for word in keyword_words):
            for chunk in menu_chunks:
                if keyword in chunk.lower() and chunk not in results:
                    results.append(chunk)

    # 3. Semantic FAISS nəticələri
    for idx in indices[0]:
        if idx < 0:
            continue

        chunk = menu_chunks[idx]
        if chunk not in results:
            results.append(chunk)

    return results[:k]


In [21]:
!pip install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [22]:
from google.colab import userdata
from openai import OpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [23]:
CHAT_MODEL = "openrouter/free"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

Chat model: openrouter/free
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [24]:
!pip install -U diffusers transformers accelerate safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 35.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0


In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

IMAGE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_DTYPE = torch.float16 if IMAGE_DEVICE == "cuda" else torch.float32

image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=IMAGE_DTYPE,
    variant="fp16" if IMAGE_DEVICE == "cuda" else None
)

image_pipe = image_pipe.to(IMAGE_DEVICE)

print("Image generation device:", IMAGE_DEVICE)


[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
image_cache = {}

In [ ]:
def generate_image(prompt_en):
    cache_key = prompt_en.strip().lower()

    if cache_key in image_cache:
        print("Image loaded from cache.")
        image = image_cache[cache_key]
        display(image)
        return image

    image = image_pipe(
        prompt=prompt_en,
        num_inference_steps=1,
        guidance_scale=0.0
    ).images[0]

    image_cache[cache_key] = image

    display(image)

    return image

In [ ]:
def create_image_prompt(user_prompt, food_context):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Create a detailed English prompt for food image generation. "
                    "Use the restaurant menu information provided by the user. "
                    "The image should look realistic and appetizing, like professional "
                    "restaurant food photography. Use the ingredients and details from "
                    "the menu information. Do not invent ingredients that are not mentioned. "
                    "Return only the English image prompt."
                )
            },
            {
                "role": "user",
                "content": f"""
User request:
{user_prompt}

Restaurant menu information:
{food_context}
"""
            }
        ]
    )

    prompt_en = response.choices[0].message.content.strip()

    return prompt_en

In [ ]:
def handle_image_request(user_prompt):
    clean_query = clean_image_query(user_prompt)

    # Bir yeməyin məlumatı bir neçə chunk-a bölünə bilər.
    # Retriever eyni Path-ə aid chunk-ları birlikdə qaytarır.
    image_chunks = menu_retriever(clean_query, k=5)
    food_context = "\n\n".join(image_chunks)

    prompt_en = create_image_prompt(
        user_prompt,
        food_context
    )

    print("English image prompt:", prompt_en)
    return generate_image(prompt_en)


In [ ]:
def rag_retriever(query, k=5):
    query_embedding = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    search_k = min(max(k * 3, 10), len(processed_chunks))
    scores, indices = index.search(query_embedding, search_k)

    results = []
    query_lower = query.lower()

    # 1. Query ilə Path arasında uyğunluq
    for chunk in processed_chunks:
        path = chunk.split("\n", 1)[0].lower()
        path_parts = [part.strip() for part in path.split(">")]

        for part in path_parts:
            words = part.split()

            if any(
                len(word) >= 5 and word in query_lower
                for word in words
            ):
                if chunk not in results:
                    results.append(chunk)
                break

    # 2. Semantic FAISS nəticələri
    for idx in indices[0]:
        if idx < 0:
            continue

        chunk = processed_chunks[idx]
        if chunk not in results:
            results.append(chunk)

    return results[:k]


In [ ]:
SYSTEM_PROMPT = """
Sən Narxana Restoranlar Şəbəkəsinin ağıllı ofisiant botusan.

Qaydalar:
1. İstifadəçi başqa dil istəməsə, Azərbaycan dilində cavab ver.
2. Restoran haqqında faktları yalnız verilmiş CONTEXT-dən götür.
3. Qiymət, faiz, vaxt, porsiya sayı və digər rəqəmləri uydurma.
4. Menyu suallarında yalnız menyu məlumatlarına əsaslan.
5. Cavab CONTEXT-də yoxdursa, məlumatın tapılmadığını açıq şəkildə bildir.
6. CONTEXT-dəki faktları dəyişdirmə və olmayan məlumat əlavə etmə.
7. Cavabları qısa, aydın və nəzakətli ver.
"""


In [ ]:
def is_menu_question(user_prompt):
    prompt_lower = user_prompt.lower()

    menu_keywords = [
        "menyu", "menu", "yemək", "yeməklər", "tərkib", "allergen",
        "vegan", "vegetarian", "qlütensiz", "plov", "şorba", "salat",
        "desert", "kabab", "qutab", "düşbərə", "dolma", "səhər yeməyi",
        "imza yeməyi"
    ]

    if any(keyword in prompt_lower for keyword in menu_keywords):
        return True

    # Qiymət sözü təkbaşına bütün sualları menyuya yönləndirməsin
    food_price_words = ["qiyməti", "neçəyə", "neçə azn", "azn"]
    return any(word in prompt_lower for word in food_price_words)


In [ ]:
def is_image_request(user_prompt):
    prompt_lower = user_prompt.lower()

    image_keywords = [
        "generate",
        "şəkil yarat",
        "şəkil yaratmaq",
        "şəklini yarat",
        "şəkil generasiya",
        "image generate",
        "create image",
        "generate image"
    ]

    return any(
        keyword in prompt_lower
        for keyword in image_keywords
    )

In [ ]:
def clean_image_query(user_prompt):
    query = user_prompt.lower()

    remove_phrases = [
        "şəklini yarat",
        "şəkil yarat",
        "şəklini hazırla",
        "şəkil hazırla",
        "şəkil generasiya et",
        "image generate",
        "generate image",
        "create image",
        "generate"
    ]

    for phrase in remove_phrases:
        query = query.replace(phrase, "")

    return query.strip()

In [ ]:
def answer_question(user_prompt, history=None):
    try:
        # Image generation
        if is_image_request(user_prompt):
            handle_image_request(user_prompt)
            return "Şəkil generasiya edildi."

        # Retrieval
        if is_menu_question(user_prompt):
            retrieved_chunks = menu_retriever(user_prompt, k=5)
        else:
            retrieved_chunks = rag_retriever(user_prompt, k=5)

        context = "\n\n".join(retrieved_chunks)

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            }
        ]

        # Son dialoqlar follow-up suallar üçün saxlanılır
        if history:
            messages.extend(history[-10:])

        messages.append(
            {
                "role": "user",
                "content": f"""
CONTEXT:
{context}

SUAL:
{user_prompt}
"""
            }
        )

        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"Xəta baş verdi: {e}"


In [ ]:
def start_chat():
    history = []

    print("Narxana ağıllı ofisiant botuna xoş gəlmisiniz!")
    print("Menyu, restoran və xidmətlər haqqında sual verə bilərsiniz.")
    print("Yeməyin şəklini yaratmaq üçün, məsələn, 'Şah plovun şəklini yarat' yazın.")
    print("Çıxmaq üçün 'exit' və ya 'quit' yazın.\n")

    while True:
        user_prompt = input("Siz: ").strip()

        if user_prompt.lower() in ["exit", "quit"]:
            print("Bot: Sağ olun! Yenidən gözləyirik.")
            break

        if not user_prompt:
            continue

        answer = answer_question(
            user_prompt,
            history=history
        )

        print("Bot:", answer)

        history.append({
            "role": "user",
            "content": user_prompt
        })
        history.append({
            "role": "assistant",
            "content": answer
        })

        # Yalnız son 5 dialoq cütünü saxla
        history = history[-10:]


In [ ]:
test_questions = [
    "Narxana neçə şəhərdə fəaliyyət göstərir?",
    "İçərişəhər filialında spirtli içki varmı?",
    "Restoranda rezervasiya necə edilir?",
    "Çatdırılmada sifariş 90 dəqiqədən çox gecikərsə nə baş verir?",
    "Nar Xalı proqramında Qızıl səviyyənin cashback faizi neçədir?",
    "Düşbərənin qiyməti və tərkibi nədir?",
    "Qlütensiz əsas yemək tövsiyə et.",
    "3-4 nəfərlik imza yeməyi tövsiyə et.",
    "Şah plovun tərkibi nədir?",
    "Restorana ev heyvanı ilə gəlmək olar?",
    "Restoranda tələbələr üçün endirim varmı?"
]


In [ ]:
for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 60}")
    print(f"TEST {i}")
    print("Sual:", question)

    answer = answer_question(question)

    print("Cavab:", answer)

In [ ]:
start_chat()

In [ ]:
# API istifadə etmədən retrieval testləri
retrieval_tests = [
    "şah plov",
    "düşbərə",
    "vegan yeməklər",
    "desertlər"
]

for query in retrieval_tests:
    print(f"\n{'=' * 60}")
    print("QUERY:", query)

    results = menu_retriever(query, k=5)

    for i, result in enumerate(results, 1):
        print(f"\n--- RESULT {i} ---")
        print(result)
